In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 1) Imports, config & data loading
# ─────────────────────────────────────────────────────────────────────────────
import os, time, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, r2_score

# suppress warnings
warnings.filterwarnings("ignore")

# CatBoost import & silence its logs
import catboost as cb

# load your splits
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te   = joblib.load("data/processed/test.pkl")

cfg = {
    "general": {
        "n_trials": 50,
        "seeds":    [0, 42, 7, 123, 999],
    },
    "catboost": {
        "iterations":            1000,
        "early_stopping_rounds": 50,
        "search_space": {
            "learning_rate": {"low": 1e-3, "high": 1e-1},
            "depth":         {"low": 3,    "high": 12},
            "subsample":     {"low": 0.5,  "high": 1.0},
            "rsm":           {"low": 0.5,  "high": 1.0},
        },
    },
}

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 2) Three CatBoost‐tuning functions w/ Optuna, Hyperopt, skopt
# ─────────────────────────────────────────────────────────────────────────────
import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import gp_minimize
from skopt.space import Real, Integer
from catboost import CatBoostRegressor, Pool

def tune_optuna_cb(seed):
    def objective(trial):
        params = {
            "iterations":         cfg["catboost"]["iterations"],
            "learning_rate":      trial.suggest_float(
                                       "learning_rate",
                                       **cfg["catboost"]["search_space"]["learning_rate"],
                                       log=True
                                   ),
            "depth":              trial.suggest_int(
                                       "depth",
                                       **cfg["catboost"]["search_space"]["depth"]
                                   ),
            "subsample":          trial.suggest_float(
                                       "subsample",
                                       **cfg["catboost"]["search_space"]["subsample"]
                                   ),
            "rsm":                trial.suggest_float(
                                       "rsm",
                                       **cfg["catboost"]["search_space"]["rsm"]
                                   ),
            "random_state":       seed,
            "allow_writing_files": False,
            "verbose":            False,
        }
        train_pool = Pool(X_tr, label=y_tr)
        val_pool   = Pool(X_val, label=y_val)
        model = CatBoostRegressor(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=cfg["catboost"]["early_stopping_rounds"],
        )
        preds = model.predict(X_val)
        return mean_squared_error(y_val, preds, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective, n_trials=cfg["general"]["n_trials"])
    best = study.best_params
    best["random_state"] = seed
    return best, study.best_value


def tune_hyperopt_cb(seed):
    space = {
        "learning_rate": hp.loguniform(
                              "learning_rate",
                              np.log(cfg["catboost"]["search_space"]["learning_rate"]["low"]),
                              np.log(cfg["catboost"]["search_space"]["learning_rate"]["high"])
                          ),
        "depth":         hp.quniform(
                              "depth",
                              cfg["catboost"]["search_space"]["depth"]["low"],
                              cfg["catboost"]["search_space"]["depth"]["high"],
                              1
                          ),
        "subsample":     hp.uniform(
                              "subsample",
                              cfg["catboost"]["search_space"]["subsample"]["low"],
                              cfg["catboost"]["search_space"]["subsample"]["high"]
                          ),
        "rsm":           hp.uniform(
                              "rsm",
                              cfg["catboost"]["search_space"]["rsm"]["low"],
                              cfg["catboost"]["search_space"]["rsm"]["high"]
                          ),
    }
    trials = Trials()

    def objective(params):
        p = {
            "iterations":         cfg["catboost"]["iterations"],
            "learning_rate":      params["learning_rate"],
            "depth":              int(params["depth"]),
            "subsample":          params["subsample"],
            "rsm":                params["rsm"],
            "random_state":       seed,
            "allow_writing_files": False,
            "verbose":            False,
        }
        train_pool = Pool(X_tr, label=y_tr)
        val_pool   = Pool(X_val, label=y_val)
        model = CatBoostRegressor(**p)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=cfg["catboost"]["early_stopping_rounds"],
        )
        preds = model.predict(X_val)
        rmse  = mean_squared_error(y_val, preds, squared=False)
        return {"loss": rmse, "status": STATUS_OK}

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=cfg["general"]["n_trials"],
        trials=trials,
        rstate=np.random.default_rng(seed)
    )
    best_rmse = np.min(trials.losses())
    return {
        "learning_rate": best["learning_rate"],
        "depth":         int(best["depth"]),
        "subsample":     best["subsample"],
        "rsm":           best["rsm"],
        "random_state":  seed,
    }, best_rmse


def tune_skopt_cb(seed):
    dims = [
        Real(cfg["catboost"]["search_space"]["learning_rate"]["low"],
             cfg["catboost"]["search_space"]["learning_rate"]["high"],
             prior="log-uniform", name="learning_rate"),
        Integer(cfg["catboost"]["search_space"]["depth"]["low"],
                cfg["catboost"]["search_space"]["depth"]["high"],
                name="depth"),
        Real(cfg["catboost"]["search_space"]["subsample"]["low"],
             cfg["catboost"]["search_space"]["subsample"]["high"],
             name="subsample"),
        Real(cfg["catboost"]["search_space"]["rsm"]["low"],
             cfg["catboost"]["search_space"]["rsm"]["high"],
             name="rsm"),
    ]
    def objective(x):
        params = {
            "iterations":         cfg["catboost"]["iterations"],
            "learning_rate":      x[0],
            "depth":              int(x[1]),
            "subsample":          x[2],
            "rsm":                x[3],
            "random_state":       seed,
            "allow_writing_files": False,
            "verbose":            False,
        }
        train_pool = Pool(X_tr, label=y_tr)
        val_pool   = Pool(X_val, label=y_val)
        model = CatBoostRegressor(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=cfg["catboost"]["early_stopping_rounds"],
        )
        preds = model.predict(X_val)
        return mean_squared_error(y_val, preds, squared=False)

    res = gp_minimize(
        objective,
        dims,
        n_calls=cfg["general"]["n_trials"],
        random_state=seed
    )
    return {
        "learning_rate": res.x[0],
        "depth":         int(res.x[1]),
        "subsample":     res.x[2],
        "rsm":           res.x[3],
        "random_state":  seed,
    }, res.fun

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 3) Run all 3 tuners → save rq1_cb_tuning.csv
# ─────────────────────────────────────────────────────────────────────────────
methods = [
    ("Optuna",   tune_optuna_cb),
    ("Hyperopt", tune_hyperopt_cb),
    ("Skopt",    tune_skopt_cb),
]

records = []
for pkg, fn in methods:
    for sd in cfg["general"]["seeds"]:
        p, rmse = fn(sd)
        records.append({
            "package":  pkg,
            "seed":     sd,
            "val_rmse": rmse,
            **p
        })

df_tune = pd.DataFrame(records)
df_tune.to_csv("rq1_cb_tuning.csv", index=False)
print("→ tuning results → rq1_cb_tuning.csv")
df_tune

[I 2025-06-09 22:59:12,991] A new study created in memory with name: no-name-74b43827-208e-4e6f-9a42-80be77b855fa
[I 2025-06-09 22:59:35,911] Trial 0 finished with value: 7872.450913306162 and parameters: {'learning_rate': 0.012520653814999465, 'depth': 10, 'subsample': 0.801381688035822, 'rsm': 0.7724415914984484}. Best is trial 0 with value: 7872.450913306162.
[I 2025-06-09 22:59:48,675] Trial 1 finished with value: 9365.663855087178 and parameters: {'learning_rate': 0.00703573702872215, 'depth': 9, 'subsample': 0.7187936056313462, 'rsm': 0.9458865003910399}. Best is trial 0 with value: 7872.450913306162.
[I 2025-06-09 23:00:00,435] Trial 2 finished with value: 6252.950104729174 and parameters: {'learning_rate': 0.08459126528049374, 'depth': 6, 'subsample': 0.8958625190413323, 'rsm': 0.7644474598764522}. Best is trial 2 with value: 6252.950104729174.
[I 2025-06-09 23:00:26,707] Trial 3 finished with value: 7431.773897718473 and parameters: {'learning_rate': 0.013680095279726922, 'dep

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [29:16<00:00, 35.12s/trial, best loss: 6189.099339199499]
→ tuning results → rq1_cb_tuning.csv


,package,seed,val_rmse,learning_rate,depth,subsample,rsm,random_state
0,Optuna,0,5880.093662,0.090074,6,0.928530,0.502015,0
1,Optuna,42,5878.498628,0.087185,6,0.720206,0.504343,42
2,Optuna,7,6018.217744,0.094561,11,0.528720,0.566634,7
3,Optuna,123,6023.665085,0.099748,8,0.843325,0.973349,123
4,Optuna,999,6055.093371,0.075669,7,0.599465,0.549606,999
5,Hyperopt,0,5991.854567,0.075284,9,0.671649,0.596205,0
6,Hyperopt,42,6021.209112,0.099811,10,0.817767,0.769879,42
7,Hyperopt,7,6124.512650,0.089600,10,0.756902,0.899304,7
8,Hyperopt,123,6101.075735,0.088316,8,0.523894,0.731440,123
9,Hyperopt,999,6189.099339,0.059691,8,0.853921,0.941351,999


In [19]:
# ─────────────────────────────────────────────────────────────
# Subtask 4: pick best CatBoost trial, retrain on (train+val), test & save
# ─────────────────────────────────────────────────────────────
import time, logging
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1) Silence CatBoost logs
logging.getLogger("catboost").setLevel(logging.CRITICAL)

# 2) Reload tuning results + your splits
df_tune = pd.read_csv("rq1_cb_tuning.csv")
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te = joblib.load("data/processed/test.pkl")

# 3) Pick the very best trial (lowest val_rmse)
best = df_tune.loc[df_tune.val_rmse.idxmin()]

# 4) Build final params dict (match your tuning-CSV columns exactly)
final_params = {
    # fixed from your cfg
    "iterations":            cfg["catboost"]["iterations"],
    "early_stopping_rounds": cfg["catboost"]["early_stopping_rounds"],
    "loss_function":         "RMSE",
    "verbose":               False,
    # tuned
    "learning_rate":     float(best["learning_rate"]),
    "depth":             int(best["depth"]),
    "subsample":         float(best["subsample"]),
    # this column in your CSV was named 'rsm' (i.e. colsample_by* → rsm)
    "colsample_bylevel": float(best["rsm"]),
    "random_seed":       int(best["random_state"]),
}

# 5) Retrain on train+val
X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])

model = CatBoostRegressor(**final_params)
t0 = time.time()
model.fit(
    X_full, y_full,
    eval_set=(X_val, y_val),
    use_best_model=True,
    verbose=False
)
train_time = time.time() - t0

# 6) Final test
t1 = time.time()
y_pred = model.predict(X_te)
pred_time = time.time() - t1

test_rmse = mean_squared_error(y_te, y_pred, squared=False)
test_r2   = r2_score(y_te, y_pred)

# 7) Summary & save
df_final = pd.DataFrame([{
    **final_params,
    "train_time_s": round(train_time, 3),
    "pred_time_s":  round(pred_time,  3),
    "test_rmse":    round(test_rmse,  3),
    "test_r2":      round(test_r2,    4)
}])

df_final.to_csv("rq1_cb_final.csv", index=False)
print("→ rq1_cb_final.csv written")
df_final

→ rq1_cb_final.csv written


,iterations,early_stopping_rounds,loss_function,verbose,learning_rate,depth,subsample,colsample_bylevel,random_seed,train_time_s,pred_time_s,test_rmse,test_r2
0,1000,50,RMSE,False,0.1,7,0.5,0.5,123,10.611,1.189,5452.723,0.9471
